# Solutions — Capstone: Bakehouse (Hard 13)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_customers    = spark.table("samples.bakehouse.sales_customers")
df_franchises   = spark.table("samples.bakehouse.sales_franchises")
df_suppliers    = spark.table("samples.bakehouse.sales_suppliers")
df_reviews      = spark.table("samples.bakehouse.media_customer_reviews")

dataset_map = {
    "sales_transactions":      df_transactions,
    "sales_customers":         df_customers,
    "sales_franchises":        df_franchises,
    "sales_suppliers":         df_suppliers,
    "media_customer_reviews":  df_reviews,
}


## Solution 1 — Dataset Summary

In [ ]:
rows = []
for name, df in dataset_map.items():
    rows.append((name, df.count(), len(df.columns)))
result_1 = spark.createDataFrame(rows, ["dataset_name", "row_count", "column_count"])


## Solution 2 — Discount Detection

In [ ]:
result_2 = (
    df_transactions
    .withColumn("expected_price",   F.col("unitPrice") * F.col("quantity"))
    .withColumn("discount_amount",  F.round(F.col("expected_price") - F.col("totalPrice"), 2))
    .withColumn("discountPercentage",
        F.round((F.col("expected_price") - F.col("totalPrice")) / F.col("expected_price") * 100, 2)
    )
    .filter(F.col("discount_amount") > 0)
    .select("transactionID", "product", "unitPrice", "quantity", "totalPrice",
            "expected_price", "discount_amount", "discountPercentage")
)


## Solution 3 — Revenue per Franchise per Date

In [ ]:
result_3 = (
    df_transactions
    .withColumn("date", F.to_date("dateTime"))
    .groupBy("date", "franchiseID")
    .agg(
        F.sum("quantity").alias("quantitySold"),
        F.round(F.sum("totalPrice"), 2).alias("revenue"),
    )
    .orderBy("date", "franchiseID")
)


## Solution 4 — Top 3 Franchises per Day as Nested Map

In [ ]:
daily_rev = (
    df_transactions
    .join(df_franchises.select("franchiseID", F.col("name").alias("franchise_name")), on="franchiseID")
    .withColumn("date", F.to_date("dateTime"))
    .groupBy("date", "franchiseID", "franchise_name")
    .agg(F.round(F.sum("totalPrice"), 2).alias("rev"))
)

w = Window.partitionBy("date").orderBy(F.col("rev").desc())

result_4 = (
    daily_rev
    .withColumn("rnk", F.rank().over(w))
    .filter(F.col("rnk") <= 3)
    .groupBy("date")
    .agg(
        F.round(F.sum("rev"), 2).alias("totalRevenue"),
        F.map_from_entries(
            F.collect_list(F.struct(F.col("franchise_name"), F.col("rev")))
        ).alias("franchises"),
    )
    .orderBy("date")
)


## Solution 5 — Customer Behaviour by Country and Store Size

In [ ]:
result_5 = (
    df_transactions
    .join(df_customers.select("customerID", F.col("country").alias("cust_country")), on="customerID")
    .join(df_franchises.select("franchiseID", F.col("size").alias("franchiseSize")), on="franchiseID")
    .groupBy(F.col("cust_country").alias("country"), "franchiseSize")
    .agg(
        F.countDistinct("customerID").alias("numberOfUniqueCustomers"),
        F.round(F.avg("totalPrice"), 2).alias("averageBasket"),
    )
    .orderBy("country", "franchiseSize")
)


## Solution 6 — Top 3 Products per Franchise with Countries

In [ ]:
w = Window.partitionBy("franchiseID").orderBy(F.col("revenue").desc())

result_6 = (
    df_transactions
    .join(df_franchises.select("franchiseID", F.col("name").alias("franchiseName")), on="franchiseID")
    .join(df_customers.select("customerID", "country"), on="customerID")
    .groupBy("franchiseName", "product")
    .agg(
        F.round(F.sum("totalPrice"), 2).alias("revenue"),
        F.collect_set("country").alias("countries"),
    )
    .withColumn("rnk", F.rank().over(
        Window.partitionBy("franchiseName").orderBy(F.col("revenue").desc())
    ))
    .filter(F.col("rnk") <= 3)
    .select("franchiseName", "product", "revenue", "countries")
    .orderBy("franchiseName", F.col("revenue").desc())
)


## Solution 7 — Franchise Reviews with Sentiment

In [ ]:
# Requires Databricks AI functions — ai_analyze_sentiment is Databricks-only
result_7 = (
    df_reviews
    .groupBy("sentiment")
    .agg(F.count("*").alias("numberReviews"))
    .orderBy(F.col("numberReviews").desc())
)
# Note: if running on Databricks, replace with:
# result_7 = (
#     df_reviews
#     .withColumn("sentiment", F.expr("ai_analyze_sentiment(reviewText)"))
#     .groupBy("sentiment")
#     .agg(F.count("*").alias("numberReviews"))
# )


## Solution 8 — Full PII-Masked Operational Dashboard

In [ ]:
result_8 = (
    df_transactions
    .join(df_customers, on="customerID")
    .join(df_franchises.select("franchiseID", F.col("name").alias("franchiseName"),
                               "city", "country"), on="franchiseID")
    .withColumn("firstName",   F.lit("***"))
    .withColumn("lastName",    F.lit("***"))
    .withColumn("emailAddress",
        F.concat(F.substring("emailAddress", 1, 1), F.lit("***@"),
                 F.regexp_extract("emailAddress", r"@(.+)$", 1))
    )
    .withColumn("maskedCard",
        F.concat(F.lit("****-****-****-"),
                 F.substring(F.cast("cardNumber", "string"), -4, 4))
    )
    .select(
        "transactionID", "dateTime", "franchiseName",
        "firstName", "lastName", "emailAddress", "maskedCard",
        "product", "quantity", "totalPrice", "paymentMethod"
    )
)
